# Compare FAISS Search Methods Across Datasets

One notebook for every dataset benchmarked by `benchmark_ann.py`
(Exact FlatIP vs HNSW vs IVF-PQ vs LSH), using the same protocol on each:

```text
1,000 held-out query vectors (seed 42)
        |
gallery vectors -> Exact FlatIP / HNSW / IVF-PQ / LSH
        |
exact top-20 = ground truth -> Recall@5/10/20 per method
```

| Key | Dataset | Model | Dim | Gallery |
|---|---|---|---:|---:|
| `coco` | COCO unlabeled2017 | ViT-B/16 | 768 | 122,403 |
| `zappos` | UT-Zap50K | ResNet18 | 512 | 49,025 |

Sections:

1. combined metrics table (both datasets in one table),
2. overlaid trade-off plots and parameter sweeps,
3. cross-dataset summary (incl. speed-up vs exact, which normalises the
   different gallery sizes),
4. **the same query searched by every method side by side**, per dataset,
5. Zappos category-level precision per method,
6. sanity checks.

Sections 1-3 only need `results/<dataset>/ann_results.json`. Sections 4-6
also need the per-query files from a local `benchmark_ann.py` run, the
embeddings, and (for 4-5) the images. Datasets missing any of these are
skipped with a note rather than failing.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# Datasets to include (keys from datasets.py).
DATASET_KEYS = ["coco", "zappos"]

# Query image ids to visualise in section 4, per dataset. None = first 3
# of the 1,000 benchmark queries.
QUERY_IDS = {"coco": None, "zappos": None}

# ANN settings shown side by side in section 4 (file stems from
# results/<dataset>/per_query/, without "_top20.npy").
SHOW_SETTINGS = {
    "hnsw_ef128": "HNSW (efSearch=128)",
    "ivfpq_np32": "IVF-PQ (nprobe=32)",
    "lsh_nb512": "LSH (nbits=512)",
}

K_DISPLAY = 10   # results shown per method (recall computed at this K)
SAVE_IMAGES = True

In [ ]:
# Find this experiment folder whether the notebook runs from its own
# folder or from the repository root, then reuse the shared registry.
EXPERIMENT_DIR = Path.cwd()
if not (EXPERIMENT_DIR / "datasets.py").exists():
    EXPERIMENT_DIR = Path.cwd() / "experiments" / "ann-comparison"
sys.path.insert(0, str(EXPERIMENT_DIR))
from datasets import DATASETS, REPO_ROOT, results_dir

runs = {}
for key in DATASET_KEYS:
    path = results_dir(key) / "ann_results.json"
    if not path.exists():
        print(f"{key}: no ann_results.json - run benchmark_ann.py --dataset {key}")
        continue
    data = json.loads(path.read_text())
    runs[key] = {
        "cfg": DATASETS[key],
        "label": DATASETS[key]["label"],
        "metadata": data["metadata"],
        "rows": data["results"],
        "per_query_dir": results_dir(key) / "per_query",
        "output_dir": results_dir(key) / "visual-comparisons",
    }

for key, run in runs.items():
    m = run["metadata"]
    print(f"{run['label']:<16} gallery {m['gallery_size']:>7,} | queries {m['num_queries']} "
          f"| {m['dimension']}-d | k_gt {m['k_gt']} | seed {m['seed']}")

In [ ]:
# Per-query detail (sections 4-6): needs a local benchmark run plus the
# embeddings. Images are optional and only gate the visual grids.
def load_detail(key, run):
    cfg = run["cfg"]
    pq_dir = run["per_query_dir"]
    if not (pq_dir / "exact_top20.npy").exists():
        return None, "no per_query/ files (run benchmark_ann.py locally)"
    if not cfg["emb"].exists():
        return None, f"embeddings missing: {cfg['emb']}"

    ids = np.load(cfg["ids"])
    detail = {
        "embeddings": np.load(cfg["emb"], mmap_mode="r"),
        "ids": ids,
        "query_rows": np.load(pq_dir / "query_rows.npy"),
        "gallery_rows": np.load(pq_dir / "gallery_rows.npy"),
        "exact_top20": np.load(pq_dir / "exact_top20.npy"),
        "image_path": None,
        "image_label": None,
        "categories": None,
    }
    detail["query_ids"] = ids[detail["query_rows"]]

    image_dir = cfg["image_dir"]
    if key == "coco" and image_dir.exists():
        # COCO filenames are 12-digit zero-padded ids.
        detail["image_path"] = lambda image_id: image_dir / f"{int(image_id):012d}.jpg"
    elif key == "zappos" and image_dir.exists():
        # UT-Zap50K has no integer ids: rows are the lexicographically
        # sorted image paths (same ordering extract_features.py used).
        paths = sorted(image_dir.rglob("*.jpg"), key=lambda p: str(p))
        assert len(paths) == len(ids), "image count does not match embedding rows"
        categories = np.array([p.relative_to(image_dir).parts[0] for p in paths])
        subcategories = np.array([p.relative_to(image_dir).parts[1] for p in paths])
        detail["image_path"] = lambda image_id: paths[int(image_id)]
        detail["image_label"] = lambda image_id: f"{categories[image_id]} / {subcategories[image_id]}"
        detail["categories"] = categories
    return detail, None

details = {}
for key, run in runs.items():
    detail, reason = load_detail(key, run)
    if detail is None:
        print(f"{run['label']}: per-query sections skipped - {reason}")
        continue
    details[key] = detail
    settings = sorted(p.name.replace("_top20.npy", "")
                      for p in run["per_query_dir"].glob("*_top20.npy"))
    images = "yes" if detail["image_path"] else f"no ({run['cfg']['image_dir']})"
    print(f"{run['label']}: per-query ready | images: {images}")
    print("  query ids (first 15):", detail["query_ids"][:15].tolist())
    print("  settings:", ", ".join(settings))

## 1. Combined comparison

All accuracy metrics treat the **exact top-K** as the relevant set:

- **Recall@K** = fraction of the exact top-K that the method also returns.
- **Precision@K** = fraction of the method's top-K that is in the exact
  top-K. Both lists hold K items, so it equals Recall@K here.
- **mAP@K** = mean over queries of AP@K = (1/K) * sum of Precision@i at
  every rank i holding a relevant result. It is rank-aware: a method that
  finds the right neighbours but orders them wrongly scores lower.

`n/a` marks results produced before precision/mAP were added - re-run
`benchmark_ann.py` for that dataset to fill them in. All searches ran
single-threaded; build times used all cores.

In [ ]:
def setting_str(row):
    params = row["parameters"]
    algo = row["algorithm"]
    if algo == "HNSW":
        return f"efSearch={params['efSearch']}"
    if algo == "IVF-PQ":
        return f"nprobe={params['nprobe']}"
    if algo == "LSH":
        return f"nbits={params['nbits']}"
    return "-"

def metric(row, field, width=6):
    # Older result files lack precision/mAP.
    value = row.get(field)
    return f"{'n/a':>{width}}" if value is None else f"{value:>{width}.3f}"

header = (
    f"{'Dataset':<16} {'Method':<13} {'Setting':<14} {'R@5':>6} {'R@10':>6} {'R@20':>6} "
    f"{'P@10':>6} {'mAP@5':>6} {'mAP@10':>6} {'mAP@20':>6} "
    f"{'ms/query':>9} {'QPS':>8} {'build(s)':>9} {'size(MB)':>9}"
)
print(header)
print("-" * len(header))
for key, run in runs.items():
    for row in run["rows"]:
        print(
            f"{run['label']:<16} {row['algorithm']:<13} {setting_str(row):<14} "
            f"{row['recall_at_5']:>6.3f} {row['recall_at_10']:>6.3f} {row['recall_at_20']:>6.3f} "
            f"{metric(row, 'precision_at_10')} {metric(row, 'map_at_5')} "
            f"{metric(row, 'map_at_10')} {metric(row, 'map_at_20')} "
            f"{row['latency_ms_per_query']:>9.3f} {row['queries_per_second']:>8.1f} "
            f"{row['build_seconds']:>9.1f} {row['index_size_mb']:>9.1f}"
        )
    print()

## 2. Trade-offs, both datasets overlaid

Colour = method, marker = dataset. Dashed vertical lines mark each
dataset's exact-search latency.

In [ ]:
colors = {
    "Exact FlatIP": "tab:gray",
    "HNSW": "tab:blue",
    "IVF-PQ": "tab:orange",
    "LSH": "tab:green",
}
markers = {"coco": "o", "zappos": "s"}
line_styles = {"coco": "--", "zappos": ":"}

fig, axes = plt.subplots(1, 3, figsize=(21, 5.5))
panels = [
    (axes[0], "latency_ms_per_query", "recall_at_10"),
    (axes[1], "index_size_mb", "recall_at_10"),
    (axes[2], "latency_ms_per_query", "map_at_10"),  # rank-aware view
]
for key, run in runs.items():
    for row in run["rows"]:
        color = colors[row["algorithm"]]
        for ax, x_field, y_field in panels:
            if row.get(y_field) is None:
                continue
            ax.scatter(row[x_field], row[y_field], color=color,
                       marker=markers[key], s=60, alpha=0.8)
            ax.annotate(setting_str(row), (row[x_field], row[y_field]),
                        textcoords="offset points", xytext=(6, 4), fontsize=7)
    exact_row = next(r for r in run["rows"] if r["algorithm"] == "Exact FlatIP")
    for ax in (axes[0], axes[2]):
        ax.axvline(exact_row["latency_ms_per_query"], color="tab:gray",
                   linestyle=line_styles[key], alpha=0.6)

legend = (
    [plt.Line2D([], [], marker="o", linestyle="", color=c, label=a) for a, c in colors.items()]
    + [plt.Line2D([], [], marker=markers[k], linestyle="", color="black",
                  label=runs[k]["label"]) for k in runs]
    + [plt.Line2D([], [], color="tab:gray", linestyle=line_styles[k],
                  label=f"exact latency ({runs[k]['label']})") for k in runs]
)
axes[0].set_xlabel("query latency (ms, 1 thread)")
axes[0].set_ylabel("Recall@10")
axes[0].set_title("Accuracy vs speed")
axes[0].legend(handles=legend, fontsize=8)
axes[1].set_xlabel("index size (MB)")
axes[1].set_ylabel("Recall@10")
axes[1].set_title("Accuracy vs memory")
axes[1].set_xscale("log")
axes[2].set_xlabel("query latency (ms, 1 thread)")
axes[2].set_ylabel("mAP@10")
axes[2].set_title("Ranking quality vs speed")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Parameter sweeps: one line per dataset.
sweep_specs = [
    ("HNSW", "efSearch"),
    ("IVF-PQ", "nprobe"),
    ("LSH", "nbits"),
]
fig, axes = plt.subplots(len(sweep_specs), 2, figsize=(13, 4 * len(sweep_specs)))
for i, (algo, param) in enumerate(sweep_specs):
    for key, run in runs.items():
        sub = [r for r in run["rows"] if r["algorithm"] == algo]
        x = [r["parameters"][param] for r in sub]
        style = dict(marker=markers[key], label=run["label"])
        axes[i][0].plot(x, [r["recall_at_10"] for r in sub], **style)
        axes[i][1].plot(x, [r["latency_ms_per_query"] for r in sub], **style)
    axes[i][0].set_ylabel("Recall@10")
    axes[i][0].set_title(f"{algo}: recall vs {param}")
    axes[i][1].set_ylabel("ms/query")
    axes[i][1].set_title(f"{algo}: latency vs {param}")
    for ax in axes[i]:
        ax.set_xlabel(param)
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 3. Cross-dataset summary

The datasets differ in gallery size, dimensionality and embedding model,
so raw latencies are **not** directly comparable. *Speed-up* = exact
latency / method latency on the same dataset, which puts both on one
scale. *Compression* = exact index size / method index size.

In [ ]:
def pick(results, algo, **params):
    for r in results:
        if r["algorithm"] == algo and all(
            r["parameters"].get(k) == v for k, v in params.items()
        ):
            return r
    return None

picks = [
    ("Exact FlatIP", "Exact FlatIP", {}),
    ("HNSW efSearch=16", "HNSW", {"efSearch": 16}),
    ("HNSW efSearch=64", "HNSW", {"efSearch": 64}),
    ("HNSW efSearch=128", "HNSW", {"efSearch": 128}),
    ("IVF-PQ nprobe=32", "IVF-PQ", {"nprobe": 32}),
    ("LSH nbits=512", "LSH", {"nbits": 512}),
]

cell = f"{'R@10':>6}{'mAP@10':>7}{'ms/q':>7}{'speedup':>8}{'compr':>7}"
print(f"{'':<18}" + " | ".join(f"{('--- ' + run['label'] + ' ---'):^35}" for run in runs.values()))
print(f"{'setting':<18}" + " | ".join(cell for _ in runs))
print("-" * (18 + 38 * len(runs)))
for label, algo, params in picks:
    parts = []
    for run in runs.values():
        exact = pick(run["rows"], "Exact FlatIP")
        r = pick(run["rows"], algo, **params)
        if r is None:
            parts.append(f"{'n/a':>35}")
            continue
        parts.append(
            f"{r['recall_at_10']:>6.3f}{metric(r, 'map_at_10', 7)}{r['latency_ms_per_query']:>7.3f}"
            f"{exact['latency_ms_per_query'] / r['latency_ms_per_query']:>7.1f}x"
            f"{exact['index_size_mb'] / r['index_size_mb']:>6.1f}x"
        )
    print(f"{label:<18}" + " | ".join(parts))

# Speed-up vs recall on one normalised axis.
fig, ax = plt.subplots(figsize=(11, 6))
for key, run in runs.items():
    exact = pick(run["rows"], "Exact FlatIP")
    for r in run["rows"]:
        ax.scatter(exact["latency_ms_per_query"] / r["latency_ms_per_query"], r["recall_at_10"],
                   color=colors[r["algorithm"]], marker=markers[key], s=70, alpha=0.8)
ax.axvline(1.0, color="tab:gray", linestyle="--", alpha=0.6)
ax.set_xscale("log")
ax.set_xlabel("speed-up vs exact search on the same dataset (log)")
ax.set_ylabel("Recall@10")
ax.set_title("Normalised speed-accuracy: " +
             " vs ".join(f"{run['label']} ({'circles' if k == 'coco' else 'squares'})"
                         for k, run in runs.items()))
ax.legend(handles=legend[: len(colors) + len(runs)], fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Same query, different methods

One grid per query: one row per method, all searching the **same query**
(first column). Each result is labelled `#rank OK/MISS id cosine` (plus
category for Zappos). **MISS** (red) means the exact top-K contains a
neighbour this method ranked something else in place of - a retrieval
error. The row label shows the method's per-query Recall@K.

In [ ]:
def top_k(key, setting_key, k):
    if setting_key == "exact":
        return details[key]["exact_top20"][:, :k]
    return np.load(runs[key]["per_query_dir"] / f"{setting_key}_top20.npy")[:, :k]

def query_position(key, query_id):
    matches = np.where(details[key]["query_ids"] == int(query_id))[0]
    return int(matches[0]) if len(matches) else None

def to_image_id(key, gallery_row):
    # gallery row -> original embedding row -> image id
    d = details[key]
    return int(d["ids"][d["gallery_rows"][int(gallery_row)]])

def exact_id_set(key, position, k):
    return {to_image_id(key, row) for row in details[key]["exact_top20"][position, :k]}

def per_query_recall(key, setting_key, position, k):
    found = {to_image_id(key, row) for row in top_k(key, setting_key, k)[position] if row >= 0}
    return len(found & exact_id_set(key, position, k)) / k

def per_query_ap(key, setting_key, position, k):
    # AP@k against the exact top-k (same definition as benchmark_ann.map_at_k).
    reference = exact_id_set(key, position, k)
    hits, precision_sum = 0, 0.0
    for rank, row in enumerate(top_k(key, setting_key, k)[position], start=1):
        if row >= 0 and to_image_id(key, row) in reference:
            hits += 1
            precision_sum += hits / rank
    return precision_sum / k

def compare_query(key, query_id):
    d = details[key]
    run = runs[key]
    position = query_position(key, query_id)
    if position is None:
        print(f"{key}: query {query_id} is not in the benchmark query set")
        return

    describe = d["image_label"] or (lambda image_id: "")
    query_vector = np.asarray(d["embeddings"][int(d["query_rows"][position])])
    query_image = Image.open(d["image_path"](query_id)).convert("RGB")
    query_label = describe(query_id)
    reference = exact_id_set(key, position, K_DISPLAY)

    method_list = [("exact", "Exact FlatIP")] + list(SHOW_SETTINGS.items())
    fig, axes = plt.subplots(
        len(method_list), K_DISPLAY + 1,
        figsize=(2.1 * (K_DISPLAY + 1), 3.0 * len(method_list)),
    )

    for r, (setting_key, label) in enumerate(method_list):
        # First column: the shared query image, labelled with the method.
        title = f"{run['label']}\nQUERY id {query_id}\n{query_label}\n{label}"
        if setting_key != "exact":
            recall = per_query_recall(key, setting_key, position, K_DISPLAY)
            ap = per_query_ap(key, setting_key, position, K_DISPLAY)
            title += f"\nrecall@{K_DISPLAY}: {recall:.2f} | AP@{K_DISPLAY}: {ap:.2f}"
        axes[r][0].imshow(query_image)
        axes[r][0].set_title(title, fontsize=9)
        axes[r][0].axis("off")

        for c, gallery_row in enumerate(top_k(key, setting_key, K_DISPLAY)[position], start=1):
            ax = axes[r][c]
            if gallery_row < 0:
                # FAISS pads with -1 when fewer than K results exist.
                ax.set_facecolor("lightgray")
                ax.set_title("not\nreturned", fontsize=8)
                ax.set_xticks([]); ax.set_yticks([])
                continue
            result_id = to_image_id(key, gallery_row)
            result_vector = np.asarray(d["embeddings"][int(d["gallery_rows"][gallery_row])])
            score = float(result_vector @ query_vector)  # cosine (normalized)
            hit = result_id in reference
            ax.imshow(Image.open(d["image_path"](result_id)).convert("RGB"))
            ax.set_title(
                f"#{c} {'OK' if hit else 'MISS'}\n{describe(result_id)}\nid {result_id} | {score:.3f}",
                fontsize=8, color="black" if hit else "red",
            )
            ax.axis("off")

    plt.tight_layout()
    if SAVE_IMAGES:
        run["output_dir"].mkdir(parents=True, exist_ok=True)
        out_path = run["output_dir"] / f"query_{query_id}_methods.png"
        plt.savefig(out_path, dpi=120, bbox_inches="tight")
        print("saved:", out_path.relative_to(REPO_ROOT))
    plt.show()

for key, d in details.items():
    if QUERY_IDS.get(key) is None:
        QUERY_IDS[key] = d["query_ids"][:3].tolist()
    if d["image_path"] is None:
        print(f"{runs[key]['label']}: images not found at {runs[key]['cfg']['image_dir']} - grids skipped")
        continue
    for query_id in QUERY_IDS[key]:
        compare_query(key, query_id)

### Per-query recall and AP across every setting

Recall@K and AP@K (the per-query term of mAP@K) of each benchmarked
setting for the queries chosen above.

In [ ]:
def all_settings(key):
    return [
        p.name.replace("_top20.npy", "")
        for p in sorted(runs[key]["per_query_dir"].glob("*_top20.npy"))
        if p.name != "exact_top20.npy"
    ]

for key in details:
    columns = ["exact"] + all_settings(key)
    for name, fn in [(f"recall@{K_DISPLAY}", per_query_recall), (f"AP@{K_DISPLAY}", per_query_ap)]:
        print(f"{runs[key]['label']} - {name}")
        print(f"{'query id':<10}" + "".join(f"{c:>12}" for c in columns))
        for query_id in QUERY_IDS[key]:
            position = query_position(key, query_id)
            if position is None:
                continue
            values = [1.0]  # exact is the reference: 1 by definition
            values += [fn(key, s, position, K_DISPLAY) for s in all_settings(key)]
            print(f"{query_id:<10}" + "".join(f"{v:>12.2f}" for v in values))
        print()

## 5. Zappos category-level retrieval quality

UT-Zap50K images carry labels from their folder structure
(`Boots / Sandals / Shoes / Slippers`). Beyond vector-space recall, this
measures how many of a method's top-10 results share **the query's
category**, averaged over all 1,000 queries. COCO unlabeled2017 has no
labels, so this section is Zappos only.

In [ ]:
def category_precision(key, setting_key, k=10):
    d = details[key]
    result_rows = top_k(key, setting_key, k)
    query_cats = d["categories"][d["query_rows"]][:, None]
    result_cats = d["categories"][d["gallery_rows"][result_rows]]
    return float((result_cats == query_cats)[result_rows >= 0].mean())

def row_for(key, setting_key):
    for r in runs[key]["rows"]:
        stem = {
            "HNSW": lambda p: f"hnsw_ef{p['efSearch']}",
            "IVF-PQ": lambda p: f"ivfpq_np{p['nprobe']}",
            "LSH": lambda p: f"lsh_nb{p['nbits']}",
        }.get(r["algorithm"])
        if stem and stem(r["parameters"]) == setting_key:
            return r
    return None

if "zappos" in details and details["zappos"]["categories"] is not None:
    print(f"{'setting':<24}{'recall@10':>10}{'category P@10':>15}")
    print("-" * 49)
    print(f"{'Exact FlatIP':<24}{'1.000':>10}{category_precision('zappos', 'exact'):>15.3f}")
    for setting_key in all_settings("zappos"):
        row = row_for("zappos", setting_key)
        print(
            f"{row['algorithm'] + ' ' + setting_str(row):<24}"
            f"{row['recall_at_10']:>10.3f}"
            f"{category_precision('zappos', setting_key):>15.3f}"
        )
else:
    print("skipped: needs Zappos per-query files, embeddings and images")

## 6. Sanity checks

In [ ]:
for key, run in runs.items():
    exact_metrics = next(r for r in run["rows"] if r["algorithm"] == "Exact FlatIP")
    assert abs(exact_metrics["recall_at_10"] - 1.0) < 1e-9, f"{key}: exact recall@10 must be 1.0"
    for k in run["metadata"].get("metric_ks", []):
        assert abs(exact_metrics[f"precision_at_{k}"] - 1.0) < 1e-9, f"{key}: exact P@{k} must be 1.0"
        assert abs(exact_metrics[f"map_at_{k}"] - 1.0) < 1e-9, f"{key}: exact mAP@{k} must be 1.0"
        for r in run["rows"]:
            # Exact top-K and method top-K both hold K items, so P@K == R@K.
            assert abs(r[f"precision_at_{k}"] - r[f"recall_at_{k}"]) < 1e-9, f"{key}: P@{k} != R@{k}"
            assert r[f"map_at_{k}"] <= r[f"recall_at_{k}"] + 1e-9, f"{key}: mAP@{k} > R@{k}"

    if key not in details:
        print(f"{run['label']}: aggregate check passed (per-query checks skipped)")
        continue

    d = details[key]
    num_queries = run["metadata"]["num_queries"]
    k_gt = run["metadata"]["k_gt"]
    gallery_size = run["metadata"]["gallery_size"]
    for path in sorted(run["per_query_dir"].glob("*_top20.npy")):
        arr = np.load(path)
        assert arr.shape == (num_queries, k_gt), f"{key}: bad shape in {path.name}: {arr.shape}"
        assert arr.min() >= -1 and arr.max() < gallery_size, f"{key}: out-of-range rows in {path.name}"

    # Held-out split: a query image must never appear in its own results.
    for position in range(num_queries):
        query_id = int(d["query_ids"][position])
        result_ids = {to_image_id(key, row) for row in d["exact_top20"][position] if row >= 0}
        assert query_id not in result_ids, f"{key}: query {query_id} leaked into gallery"

    print(f"{run['label']}: ALL CHECKS PASSED "
          f"(shapes {(num_queries, k_gt)}, exact recall 1.0, held-out split verified)")

## How to adjust

- **Fewer datasets**: edit `DATASET_KEYS` in the config cell.
- **Other queries**: set `QUERY_IDS["coco"]` / `QUERY_IDS["zappos"]` to ids
  printed in the setup cell, then re-run section 4.
- **Other settings side by side**: replace keys in `SHOW_SETTINGS` with any
  listed setting (e.g. `"hnsw_ef256"`).
- **More displayed results**: raise `K_DISPLAY` (up to 20, the saved
  ground-truth depth).
- **New dataset**: add an entry to `datasets.py`, run
  `benchmark_ann.py --dataset <key>`, and add the key to `DATASET_KEYS`.
- The notebook only reads benchmark outputs; it never recomputes FAISS or
  the embedding models.